<a href="https://colab.research.google.com/github/TBNRGarret/signsight/blob/main/notebooks/SignSight_SupSign_YOLO11s_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SignSight — EDA + train SupSign 35 lớp, YOLO11s

Fine-tune **YOLO11s pretrained** trên SupSign 107 lớp, xuất taxonomy 35 lớp dành cho ô tô.
Chọn **Runtime → Change runtime type → GPU**, rồi chạy các cell theo thứ tự.

**Sửa SOURCE_ZIP và RUN_NAME ở cell Cấu hình; xem EDA rồi bật LABEL_REVIEWED để train.** Đặt ZIP SupSign lên Drive một lần.
Notebook chứa sẵn toàn bộ code chuẩn bị/train; không cần clone repo hay đăng nhập dịch vụ dataset.
Ảnh train được đọc từ SSD local Colab; dataset đã chuẩn bị và checkpoint giữ trên Drive qua các phiên.

Cả hai ZIP từng khảo sát có ảnh trùng giữa split. Notebook bỏ split nguồn, kiểm tra toàn bộ ảnh/nhãn,
loại trùng SHA-256, cách ly nhãn lỗi hoặc bản sao khác nhãn rồi chia theo nhóm.
Nếu không có group CSV, dùng dHash + block tên frame làm **heuristic phát triển**.
Điểm test lúc đó chưa chứng minh hiệu quả trên chuyến xe/địa điểm độc lập.

Nguồn API: [YOLO11](https://docs.ultralytics.com/models/yolo11/), [Train/resume](https://docs.ultralytics.com/modes/train/).

In [1]:
import sys, subprocess, os
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'ultralytics==8.4.173', 'PyYAML>=6,<7', 'tqdm>=4.66,<5'])
import torch
assert torch.cuda.is_available(), 'Bật GPU runtime trước khi train.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

GPU: Tesla T4 | Torch: 2.11.0+cu130


## 1. Cấu hình và Google Drive

Đường dẫn mặc định tương ứng: **My Drive / SignSight / datasets / archive (1).zip**.
Tên ZIP có thể khác; sửa SOURCE_ZIP cho đúng. Không dùng archive.zip 100 lớp thay vào mà giữ nhầm tên nguồn.

Mặc định: 80 epochs, ảnh 640, batch 8. Nếu hết VRAM, bắt đầu một run mới với batch 4.
Giữ 640 làm mốc trước; thử 960 trong run khác để so sánh trên cùng split.
Đổi **DATASET_VERSION** khi thay ZIP nguồn; cache cũ chỉ được tái sử dụng cho cùng phiên bản nguồn.
GROUPS_CSV tùy chọn có cột `path,group`; path là đường dẫn ảnh bên trong ZIP, ví dụ
`train/images/33292.png,trip_01`. Khi cung cấp, phải điền nhóm cho mọi ảnh hợp lệ được giữ lại, kể cả bản sao.
Nhóm cần dựa trên thông tin thu thập thật. Không tự coi tên file là metadata chuyến xe.

In [2]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

SOURCE_ZIP = Path('/content/drive/MyDrive/SignSight/datasets/archive (1).zip')
PERSIST = Path('/content/drive/MyDrive/SignSight/supsign_training')
WORK = Path('/content/signsight_supsign')
DATASET_VERSION = 'supsign107-car35-v3'
RUN_NAME = 'yolo11s_car35_640_v3'
GROUPS_CSV = None  # hoặc Path('/content/drive/MyDrive/SignSight/capture_groups.csv')
EPOCHS = 80
IMAGE_SIZE = 640
BATCH = 8
RESUME = False  # True để tiếp tục RUN_NAME cũ từ last.pt, với cùng cấu hình
SMOKE = False   # True chỉ thử 1 epoch; không dùng run smoke để kết luận chất lượng
LABEL_REVIEWED = False  # Sau khi xem EDA/ảnh bbox bên dưới, đổi True để bắt đầu train
RUN_HELDOUT_TEST = False  # chỉ bật sau khi chốt model/settings bằng validation
SEED = 42
FRAME_BLOCK_SIZE = 100
NEAR_HASH_DISTANCE = 4  # dHash: nhóm ứng viên ảnh gần giống, không chứng minh cùng chuyến

assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
assert DATASET_VERSION and Path(DATASET_VERSION).name == DATASET_VERSION and DATASET_VERSION not in ('.', '..')
PERSIST.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(WORK / '.ultralytics')
CACHE_ZIP = PERSIST / 'datasets' / (DATASET_VERSION + '.zip')
DATASET_ROOT = WORK / DATASET_VERSION
RUN_DIR = PERSIST / 'runs' / (RUN_NAME + ('_smoke' if SMOKE else ''))
print('Input:', SOURCE_ZIP)
print('Cache:', CACHE_ZIP)
print('Run:', RUN_DIR)

Mounted at /content/drive
Input: /content/drive/MyDrive/SignSight/datasets/archive (1).zip
Cache: /content/drive/MyDrive/SignSight/supsign_training/datasets/supsign107-car35-v3.zip
Run: /content/drive/MyDrive/SignSight/supsign_training/runs/yolo11s_car35_640_v3


In [3]:
# Code đã kiểm thử, nhúng trong notebook để chạy độc lập.
import json, shutil, csv, zipfile
TRAINING_FILES = {'scripts/dataset_eda.py': '"""Read-only detection EDA, including files inside ZIPs. No training or remapping."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nIMAGE_EXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\n\n\nclass DatasetReader:\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        if self.root.is_file() and self.root.suffix.lower() == \'.zip\':\n            paths, base = [self.root], self.root.parent\n        elif self.root.is_dir():\n            paths, base = sorted(self.root.rglob(\'*\')), self.root\n        else:\n            raise FileNotFoundError(f\'Dataset directory or ZIP not found: {self.root}\')\n        self.entries, self.zips, self.warnings = {}, {}, []\n        for path in paths:\n            if not path.is_file() or path.is_symlink():\n                continue\n            name = path.relative_to(base).as_posix()\n            self.entries[name] = (path, None, path.stat().st_size)\n            if path.suffix.lower() == \'.zip\':\n                try:\n                    archive = zipfile.ZipFile(path)\n                    self.zips[name] = archive\n                    for item in archive.infolist():\n                        if not item.is_dir():\n                            self.entries[name + \'!/\' + item.filename] = (archive, item, item.file_size)\n                            if PurePosixPath(item.filename).suffix.lower() in {\'.zip\', \'.rar\', \'.7z\', \'.gz\', \'.tar\'}:\n                                self.warnings.append(f\'Nested archive not expanded: {name}!/{item.filename}\')\n                except (OSError, zipfile.BadZipFile) as error:\n                    self.warnings.append(f\'Cannot open archive {name}: {type(error).__name__}\')\n\n    def open(self, name):\n        source, member, _ = self.entries[name]\n        return source.open(member) if member is not None else source.open(\'rb\')\n\n    def read(self, name, max_bytes=256 * 1024 * 1024):\n        if self.entries[name][2] > max_bytes:\n            raise ValueError(f\'Annotation exceeds {max_bytes // 1024**2} MiB read limit\')\n        with self.open(name) as stream:\n            value = stream.read(max_bytes + 1)\n        if len(value) > max_bytes:\n            raise ValueError(\'Annotation read limit exceeded\')\n        return value\n\n    def close(self):\n        for archive in self.zips.values():\n            archive.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *_):\n        self.close()\n\n\ndef split_name(name):\n    mapping = {\'train\': \'train\', \'training\': \'train\', \'val\': \'val\', \'valid\': \'val\',\n               \'validation\': \'val\', \'test\': \'test\', \'testing\': \'test\'}\n    found = {mapping[part.casefold()] for part in PurePosixPath(name).parts[:-1] if part.casefold() in mapping}\n    return next(iter(found)) if len(found) == 1 else \'unknown\'\n\n\ndef paired_files(name, extension, directory_pairs, case_names):\n    path = PurePosixPath(name)\n    candidates = {str(path.with_suffix(extension))}\n    parts = list(path.parts)\n    for i, part in enumerate(parts[:-1]):\n        for before, after in directory_pairs:\n            if part.casefold() == before.casefold():\n                changed = parts.copy(); changed[i] = after\n                candidates.add(str(PurePosixPath(*changed).with_suffix(extension)))\n    return sorted({actual for candidate in candidates for actual in case_names.get(candidate.casefold(), [])})\n\n\ndef audit_dataset(reader, dataset, image_limit=None, hash_limit=2000, seed=42,\n                  verify_decode=False, progress=None, annotation_format=\'auto\'):\n    """All counts refer to scanned images; partial/hash coverage is explicit.\n\n    Zero objects means empty_annotation only for a recognized annotation file.\n    Missing/unreadable/ambiguous labels never become verified background images.\n    """\n    if annotation_format not in {\'auto\', \'yolo\', \'voc\', \'coco\'}:\n        raise ValueError(\'annotation format must be auto, yolo, voc or coco\')\n    rng = random.Random(seed)\n    images = sorted(name for name in reader.entries if PurePosixPath(name).suffix.lower() in IMAGE_EXTENSIONS)\n    if (image_limit is not None and image_limit < 0) or (hash_limit is not None and hash_limit < 0):\n        raise ValueError(\'Limits must be nonnegative or None\')\n    selected = images if image_limit is None else sorted(rng.sample(images, min(image_limit, len(images))))\n    hashed = set(selected if hash_limit is None else rng.sample(selected, min(hash_limit, len(selected))))\n    issues, image_rows, boxes, catalogs = [], [], [], []\n    case_names = defaultdict(list)\n    for name in reader.entries:\n        case_names[name.casefold()].append(name)\n\n    def issue(path, message):\n        issues.append({\'dataset\': dataset, \'path\': path, \'message\': str(message)[:600]})\n\n    for warning in reader.warnings:\n        issue(\'\', warning)\n    for name in reader.entries:\n        suffix = PurePosixPath(name).suffix.lower()\n        try:\n            names = None\n            if suffix in {\'.yaml\', \'.yml\'}:\n                config = yaml.safe_load(reader.read(name, 1024**2))\n                if isinstance(config, dict) and \'names\' in config:\n                    names = config[\'names\']\n            elif PurePosixPath(name).name.casefold() in {\'classes.txt\', \'obj.names\', \'classes.names\'}:\n                names = reader.read(name, 1024**2).decode(\'utf-8-sig\').splitlines()\n            if names is not None:\n                if isinstance(names, list):\n                    names = dict(enumerate(names))\n                if not isinstance(names, dict) or not names:\n                    raise ValueError(\'Invalid class names\')\n                normalized = {int(key): str(value).strip() for key, value in names.items()}\n                if any(key < 0 or not value for key, value in normalized.items()):\n                    raise ValueError(\'Invalid class ID/name\')\n                catalogs.append((name, normalized))\n        except Exception as error:\n            issue(name, f\'Class metadata: {type(error).__name__}: {error}\')\n\n    def catalog_for(image_name):\n        matches = [(name, names) for name, names in catalogs\n                   if str(PurePosixPath(name).parent) == \'.\'\n                   or image_name.startswith(str(PurePosixPath(name).parent) + \'/\')]\n        if matches:\n            depth = max(len(PurePosixPath(name).parent.parts) for name, _ in matches)\n            closest = [(name, names) for name, names in matches if len(PurePosixPath(name).parent.parts) == depth]\n            if len({json.dumps(names, sort_keys=True) for _, names in closest}) > 1:\n                issue(image_name, \'Conflicting class maps at the same directory; class IDs remain unmapped.\')\n                return (\'ambiguous-class-map\', {})\n            return closest[0]\n        unique = {json.dumps(names, sort_keys=True) for _, names in catalogs}\n        return catalogs[0] if len(unique) == 1 else (\'unmapped\', {})\n\n    # COCO indexing: resolve full/suffix names; ambiguous duplicate basenames are errors.\n    by_basename = defaultdict(list)\n    for name in images:\n        by_basename[PurePosixPath(name).name].append(name)\n    coco = defaultdict(list)\n    for name in reader.entries:\n        if annotation_format not in {\'auto\', \'coco\'} or PurePosixPath(name).suffix.lower() != \'.json\':\n            continue\n        try:\n            content = json.loads(reader.read(name))\n            if not isinstance(content, dict) or not {\'images\', \'annotations\', \'categories\'} <= content.keys():\n                issue(name, \'JSON is not COCO detection; inspect separately (e.g. LabelMe/classification).\')\n                continue\n            categories = {row[\'id\']: str(row[\'name\']) for row in content[\'categories\']}\n            annotations = defaultdict(list)\n            for annotation in content[\'annotations\']:\n                annotations[annotation[\'image_id\']].append(annotation)\n            for image in content[\'images\']:\n                filename = str(image[\'file_name\']).replace(\'\\\\\', \'/\')\n                candidates = by_basename[PurePosixPath(filename).name]\n                suffix_matches = [key for key in candidates if key == filename or key.endswith(\'/\' + filename)]\n                if suffix_matches:\n                    candidates = suffix_matches\n                # Do not join a JSON in one archive to same-named images in another.\n                container = name.split(\'!/\', 1)[0] if \'!/\' in name else None\n                candidates = [key for key in candidates if\n                              (key.startswith(container + \'!/\') if container else \'!/\' not in key)]\n                if len(candidates) != 1:\n                    issue(name, f\'COCO image reference missing/ambiguous: {filename}\')\n                    continue\n                coco[candidates[0]].append((name, categories, image, annotations[image[\'id\']]))\n        except Exception as error:\n            issue(name, f\'JSON: {type(error).__name__}: {error}\')\n\n    iterator = progress(selected, desc=dataset, unit=\'image\') if progress else selected\n    for name in iterator:\n        row = {\'dataset\': dataset, \'path\': name, \'split\': split_name(name), \'parent_folder\': str(PurePosixPath(name).parent),\n               \'width\': None, \'height\': None, \'mode\': None, \'format\': None,\n               \'annotation_status\': \'missing_annotation\', \'boxes\': 0, \'sha256\': None}\n        image_rows.append(row)\n        try:\n            with reader.open(name) as stream, Image.open(stream) as image:\n                row.update(width=image.width, height=image.height, mode=image.mode)\n                if verify_decode:\n                    image.load()\n            if name in hashed:\n                digest = hashlib.sha256()\n                with reader.open(name) as stream:\n                    for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                        digest.update(block)\n                row[\'sha256\'] = digest.hexdigest()\n        except Exception as error:\n            row[\'annotation_status\'] = \'unreadable_image\'\n            issue(name, f\'Image: {type(error).__name__}: {error}\')\n            continue\n        txt = paired_files(name, \'.txt\', [(\'images\', \'labels\')], case_names)\n        xml = paired_files(name, \'.xml\', [(\'JPEGImages\', \'Annotations\'), (\'images\', \'annotations\')], case_names)\n        options = [(\'yolo\', path) for path in txt] + [(\'voc\', path) for path in xml] + [(\'coco\', entry) for entry in coco[name]]\n        if annotation_format != \'auto\':\n            options = [option for option in options if option[0] == annotation_format]\n        if not options:\n            continue\n        if len(options) > 1:\n            row[\'annotation_status\'] = \'ambiguous_annotation\'\n            issue(name, \'Multiple matching annotations/formats; select a single export/root before training.\')\n            continue\n        kind, annotation = options[0]\n        row[\'format\'] = kind\n        raw_boxes, bad = [], False\n        try:\n            if kind == \'yolo\':\n                source, names = catalog_for(name)\n                for line_number, line in enumerate(reader.read(annotation, 8 * 1024**2).decode(\'utf-8-sig\').splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        fields = line.split()\n                        if len(fields) != 5:\n                            raise ValueError(\'YOLO detection expects 5 columns; segmentation/other format is not inferred\')\n                        class_id = int(fields[0])\n                        if class_id < 0 or names and class_id not in names:\n                            raise ValueError(\'Class ID outside class map\')\n                        cx, cy, width, height = map(float, fields[1:])\n                        raw_boxes.append((str(class_id), names.get(class_id, f\'class_{class_id}\'), bool(names),\n                                          source, cx - width/2, cy - height/2, width, height, 0))\n                    except Exception as error:\n                        bad = True; issue(annotation, f\'Line {line_number}: {error}\')\n            elif kind == \'voc\':\n                tree = ET.fromstring(reader.read(annotation, 8 * 1024**2))\n                if tree.tag != \'annotation\':\n                    raise ValueError(\'Not a VOC annotation\')\n                for dimension in (\'width\', \'height\'):\n                    value = tree.findtext(\'size/\' + dimension)\n                    if value is not None and int(value) != row[dimension]:\n                        raise ValueError(\'VOC annotation size differs from image\')\n                for obj in tree.findall(\'object\'):\n                    label = obj.findtext(\'name\')\n                    if not label:\n                        raise ValueError(\'VOC object without class name\')\n                    xmin, ymin, xmax, ymax = [float(obj.findtext(\'bndbox/\' + key)) for key in (\'xmin\', \'ymin\', \'xmax\', \'ymax\')]\n                    raw_boxes.append((label, label, True, \'voc:names\', xmin/row[\'width\'], ymin/row[\'height\'],\n                                      (xmax-xmin)/row[\'width\'], (ymax-ymin)/row[\'height\'], 0))\n            else:\n                annotation_path, categories, image, objects = annotation\n                for dimension in (\'width\', \'height\'):\n                    if dimension in image and int(image[dimension]) != row[dimension]:\n                        raise ValueError(\'COCO annotation size differs from image\')\n                for obj in objects:\n                    class_id = obj[\'category_id\']\n                    if class_id not in categories:\n                        raise ValueError(\'COCO category missing from categories\')\n                    x, y, width, height = map(float, obj[\'bbox\'])\n                    raw_boxes.append((str(class_id), categories[class_id], True, annotation_path,\n                                      x/row[\'width\'], y/row[\'height\'], width/row[\'width\'], height/row[\'height\'], int(obj.get(\'iscrowd\', 0))))\n            for class_id, label, mapped, schema, x, y, width, height, crowd in raw_boxes:\n                if not all(math.isfinite(v) for v in (x, y, width, height)) or min(width, height) <= 0 or min(x, y) < -1e-6 or max(x+width, y+height) > 1+1e-6:\n                    bad = True; issue(name, \'Invalid bbox: nonfinite, empty, negative, or outside image\'); continue\n                boxes.append({\'dataset\': dataset, \'path\': name, \'split\': row[\'split\'], \'format\': kind,\n                              \'label_namespace\': schema, \'class_id\': class_id, \'class_name\': label, \'class_mapped\': mapped,\n                              \'x_norm\': x, \'y_norm\': y, \'width_norm\': width, \'height_norm\': height,\n                              \'area_fraction\': width * height, \'iscrowd\': crowd,\n                              \'short_side_at_640\': min(width*row[\'width\'], height*row[\'height\']) * 640 / max(row[\'width\'], row[\'height\'])})\n                row[\'boxes\'] += 1\n            row[\'annotation_status\'] = \'invalid_annotation\' if bad else \'annotated\' if row[\'boxes\'] else \'empty_annotation\'\n        except Exception as error:\n            row[\'annotation_status\'] = \'invalid_annotation\'\n            issue(name, f\'{kind}: {type(error).__name__}: {error}\')\n\n    summary = {\'dataset\': dataset, \'root\': str(reader.root), \'images_total\': len(images), \'images_scanned\': len(image_rows),\n               \'partial_scan\': len(selected) < len(images), \'images_hashed\': sum(bool(row[\'sha256\']) for row in image_rows),\n               \'hash_partial\': sum(bool(row[\'sha256\']) for row in image_rows) < len(images),\n               \'verify_decode\': verify_decode, \'annotation_format\': annotation_format,\n               \'valid_boxes\': len(boxes), \'issues\': len(issues),\n               \'invalid_annotations\': sum(row[\'annotation_status\'] == \'invalid_annotation\' for row in image_rows),\n               \'annotation_status\': dict(Counter(row[\'annotation_status\'] for row in image_rows)),\n               \'formats\': dict(Counter(row[\'format\'] for row in image_rows if row[\'format\'])),\n               \'file_extensions\': dict(Counter(PurePosixPath(name).suffix.lower() for name in reader.entries)),\n               \'class_maps\': [{\'path\': name, \'names\': names} for name, names in catalogs]}\n    return {\'summary\': summary, \'images\': image_rows, \'boxes\': boxes, \'issues\': issues}\n\n\ndef duplicate_groups(image_rows):\n    grouped = defaultdict(list)\n    for row in image_rows:\n        if row.get(\'sha256\'):\n            grouped[row[\'sha256\']].append(row)\n    return [{\'sha256\': digest, \'copies\': len(rows), \'cross_dataset\': len({r[\'dataset\'] for r in rows}) > 1,\n             \'cross_split\': len({r[\'split\'] for r in rows} - {\'unknown\'}) > 1,\n             \'members\': [{\'dataset\': r[\'dataset\'], \'path\': r[\'path\'], \'split\': r[\'split\']} for r in rows]}\n            for digest, rows in grouped.items() if len(rows) > 1]\n', 'scripts/supsign_training.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom scripts.dataset_eda import DatasetReader, audit_dataset\n\nCAR35 = [\n    \'Speed_limit_40\', \'Speed_limit_50\', \'Speed_limit_60\', \'Speed_limit_70\', \'Speed_limit_80\',\n    \'No_entry\', \'No_cars\', \'No_cars_turn_left\', \'No_cars_turn_right\',\n    \'No_turn_left\', \'No_turn_right\', \'No_U_turn\', \'No_cars_U_turn\',\n    \'No_straight_ahead\', \'No_turn_left_and_U_turn\', \'No_turn_left_and_right\',\n    \'No_parking\', \'No_stopping_and_parking\', \'No_parking_odd_days\', \'No_parking_even_days\',\n    \'Go_straight\', \'Keep_right\', \'Roundabout\', \'One_way_road\', \'Road_reserved_for_cars\',\n    \'Pedestrian_crossing\', \'Children_crossing\', \'Road_works\', \'Slow_down\',\n    \'Dangerous_curve_left\', \'Dangerous_curve_right\',\n    \'No_overtaking\', \'End_no_overtaking\', \'Give_way\', \'End_all_prohibitions\',\n]\nSPLITS = (\'train\', \'val\', \'test\')\nVERSION = 1\n\n\ndef digest_file(path):\n    with Path(path).open(\'rb\') as stream:\n        digest = hashlib.sha256()\n        for block in iter(lambda: stream.read(1024*1024), b\'\'): digest.update(block)\n        return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef recipe(classes=CAR35, seed=42, block_size=100, near_threshold=4, groups=None):\n    # Implementation hash invalidates caches when embedded preparation code changes.\n    group_digest = hashlib.sha256(json.dumps(groups, sort_keys=True).encode()).hexdigest()\n    return dict(version=VERSION, classes=list(classes), seed=seed, block_size=block_size,\n                near_threshold=near_threshold, groups_sha256=group_digest,\n                implementation_sha256=digest_file(__file__),\n                reader_sha256=digest_file(Path(__file__).with_name(\'dataset_eda.py\')))\n\n\ndef visual_hash(image):\n    grey = image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS)\n    pixels = list(grey.tobytes()); result = 0\n    for y in range(8):\n        for x in range(8): result = (result << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n    return result\n\n\ndef cluster_records(rows, *, groups=None, block_size=100, near_threshold=4, progress=None):\n    """Union supplied captures, filename blocks and dHash neighbours transitively.\n\n    Filename blocks / visual hashes are only development heuristics, never proof of\n    independent trips or physical signs. Explicit groups replace filename blocks.\n    """\n    if block_size < 1 or (near_threshold is not None and not 0 <= near_threshold <= 8):\n        raise ValueError(\'Invalid grouping settings\')\n    parent = list(range(len(rows)))\n    def find(i):\n        while i != parent[i]: parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    def union(a, b):\n        a, b = find(a), find(b)\n        if a != b: parent[max(a,b)] = min(a,b)\n    seen = {}\n    tree = None\n    grouping = progress(rows, desc=\'Group related images\', unit=\'image\') if progress else rows\n    for index, row in enumerate(grouping):\n        for alias in row[\'aliases\']:\n            relative = alias.split(\'!/\', 1)[-1]\n            if groups is not None:\n                group = groups.get(relative, groups.get(alias))\n                if not isinstance(group, str) or not group.strip():\n                    raise ValueError(f\'Missing capture group: {relative}\')\n                key = (\'capture\', group.strip())\n            else:\n                match = re.fullmatch(r\'(.*?)([0-9]+)\', PurePosixPath(relative).stem)\n                key = (\'filename\', match[1], int(match[2]) // block_size) if match else (\'image\', row[\'sha256\'])\n            if key in seen: union(index, seen[key])\n            else: seen[key] = index\n        if near_threshold is None: continue\n        value = row[\'visual_hash\']\n        if tree is None: tree = (value, index, {}); continue\n        # BK tree lookup: no O(n^2) all-pairs hash matrix.\n        pending = [tree]\n        while pending:\n            other, representative, children = pending.pop()\n            distance = (value ^ other).bit_count()\n            if distance <= near_threshold: union(index, representative)\n            pending.extend(child for edge, child in children.items()\n                           if distance-near_threshold <= edge <= distance+near_threshold)\n        node = tree\n        while True:\n            distance = (value ^ node[0]).bit_count()\n            if distance == 0: break\n            if distance not in node[2]: node[2][distance] = (value, index, {}); break\n            node = node[2][distance]\n    return [f\'g{find(i):06d}\' for i in range(len(rows))]\n\n\ndef choose_splits(rows, names, seed):\n    grouped = defaultdict(list)\n    for row in rows: grouped[row[\'group\']].append(row)\n    if len(grouped) < 3: raise ValueError(\'Need at least 3 independent candidate groups; provide capture groups\')\n    keys = sorted(grouped)\n    group_counts = {key: Counter(int(line.split()[0]) for row in values for line in row[\'lines\'])\n                    for key, values in grouped.items()}\n    total = sum(group_counts.values(), Counter())\n    absent = [names[i] for i in range(len(names)) if not total[i]]\n    if absent: raise ValueError(\'No usable training annotations for classes: \'+str(absent))\n    support = {i: {key for key in keys if group_counts[key][i]} for i in range(len(names))}\n    forced_train = {next(iter(groups)) for groups in support.values() if len(groups) == 1}\n    eligible = [key for key in keys if key not in forced_train]\n    # Removing two holdout groups loses coverage only for a class supported by\n    # exactly those two groups. This finds a feasible seed pair without splitting\n    # a group or depending on forty random attempts to rescue a rare class.\n    incompatible = {frozenset(groups) for groups in support.values() if len(groups) == 2}\n    ratios = {\'train\': .7, \'val\': .15, \'test\': .15}\n    def balance(sizes):\n        return sum(abs(sizes[s]/len(rows)-ratios[s]) for s in SPLITS)\n    best = None\n    for trial in range(40):\n        rng = random.Random(seed+trial)\n        candidates = eligible.copy(); rng.shuffle(candidates)\n        pair = next(((a,b) for i,a in enumerate(candidates) for b in candidates[i+1:]\n                     if frozenset((a,b)) not in incompatible), None)\n        if pair is None:\n            raise ValueError(\'Cannot preserve train class coverage with non-empty val/test groups; \'\n                             \'provide more independent capture groups, or explicitly revise the taxonomy\')\n        assignments = dict.fromkeys(keys, \'train\')\n        sizes = Counter(train=len(rows))\n        counts = {\'train\': total.copy(), \'val\': Counter(), \'test\': Counter()}\n        for key, split in zip(pair, (\'val\',\'test\')):\n            assignments[key] = split\n            sizes[\'train\'] -= len(grouped[key]); sizes[split] += len(grouped[key])\n            counts[\'train\'].subtract(group_counts[key]); counts[split].update(group_counts[key])\n        order = keys.copy(); rng.shuffle(order)\n        order.sort(key=lambda k: -len(grouped[k]))\n        missing_count = sum(not counts[s][i] for s in (\'val\',\'test\') for i in range(len(names)))\n        score = missing_count*1e6 + balance(sizes)\n        for key in order:\n            gc = group_counts[key]\n            if assignments[key] != \'train\' or any(counts[\'train\'][c] <= n for c,n in gc.items()):\n                continue\n            choices = []\n            for split in (\'val\',\'test\'):\n                proposed = sizes.copy()\n                proposed[\'train\'] -= len(grouped[key]); proposed[split] += len(grouped[key])\n                remaining = missing_count-sum(not counts[split][c] for c in gc)\n                choices.append((remaining*1e6+balance(proposed), split, remaining, proposed))\n            cost, split, remaining, proposed = min(choices, key=lambda v: v[0])\n            if cost < score:\n                assignments[key] = split; sizes = proposed; missing_count = remaining; score = cost\n                counts[\'train\'].subtract(gc); counts[split].update(gc)\n        missing = {s: [i for i in range(len(names)) if not counts[s][i]] for s in SPLITS}\n        if best is None or score < best[0]: best = score, assignments, counts, missing\n    _, assignments, counts, missing = best\n    if missing[\'train\']: raise ValueError(\'Missing train classes after grouping: \'+str([names[i] for i in missing[\'train\']]))\n    for row in rows: row[\'split\'] = assignments[row[\'group\']]\n    return counts, {s: [names[i] for i in missing[s]] for s in SPLITS}\n\n\ndef prepare_supsign(source, output, *, classes=CAR35, groups=None, seed=42,\n                    block_size=100, near_threshold=4, progress=None):\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if not source.is_file() or not zipfile.is_zipfile(source): raise ValueError(\'SOURCE_ZIP must be an existing YOLO ZIP\')\n    if output.exists(): raise FileExistsError(output)\n    if not classes or len(set(classes)) != len(classes): raise ValueError(\'Choose unique target classes\')\n    settings = recipe(classes, seed, block_size, near_threshold, groups)\n    excluded, rows, by_hash = [], [], defaultdict(list)\n    with DatasetReader(source) as reader:\n        if reader.warnings: raise ValueError(\'Source archive warnings: \'+str(reader.warnings[:3]))\n        report = audit_dataset(reader, \'supsign\', hash_limit=None, verify_decode=True,\n                               annotation_format=\'yolo\', progress=progress)\n        source_names = {name for catalog in report[\'summary\'][\'class_maps\'] for name in catalog[\'names\'].values()}\n        missing = set(classes)-source_names\n        if missing: raise ValueError(\'Missing source classes: \'+str(sorted(missing)))\n        boxes = defaultdict(list)\n        for box in report[\'boxes\']: boxes[box[\'path\']].append(box)\n        for image in report[\'images\']:\n            status = image[\'annotation_status\']\n            if status not in (\'annotated\',\'empty_annotation\'):\n                excluded.append(dict(source_path=image[\'path\'],reason=status));continue\n            if any(not b[\'class_mapped\'] for b in boxes[image[\'path\']]):\n                raise ValueError(\'Unmapped source class metadata\')\n            by_hash[image[\'sha256\']].append(image)\n        duplicates_removed = 0\n        candidates = sorted(by_hash.items())\n        if progress: candidates = progress(candidates, desc=\'Deduplicate / visual hashes\', unit=\'image\')\n        for digest, copies in candidates:\n            # Compare ALL source labels, not just selected classes. Disagreement is quarantined.\n            signatures = {tuple(sorted((b[\'class_name\'], *(round(b[k],8) for k in\n                          [\'x_norm\',\'y_norm\',\'width_norm\',\'height_norm\'])) for b in boxes[v[\'path\']])) for v in copies}\n            if len(signatures) != 1:\n                excluded.extend(dict(source_path=v[\'path\'],reason=\'conflicting_duplicate_labels\') for v in copies);continue\n            representative = min(copies,key=lambda v:v[\'path\']); lines = []\n            for b in boxes[representative[\'path\']]:\n                if b[\'class_name\'] not in classes:continue\n                x,y = b[\'x_norm\']+b[\'width_norm\']/2,b[\'y_norm\']+b[\'height_norm\']/2\n                lines.append(f"{classes.index(b[\'class_name\'])} {x:.10f} {y:.10f} {b[\'width_norm\']:.10f} {b[\'height_norm\']:.10f}")\n            with reader.open(representative[\'path\']) as stream, Image.open(stream) as image:\n                value = visual_hash(image) if near_threshold is not None else 0\n            rows.append(dict(source_path=representative[\'path\'],aliases=sorted(v[\'path\'] for v in copies),\n                             sha256=digest,lines=lines,visual_hash=value))\n            duplicates_removed += len(copies)-1\n        if not rows: raise ValueError(\'No usable images remain\')\n        for row, group in zip(rows,cluster_records(rows,groups=groups,block_size=block_size,\n                                                 near_threshold=near_threshold,progress=progress)):\n            row[\'group\'] = group\n        report[\'_split_provenance\'] = \'provided_capture_groups\' if groups is not None else \'development_heuristic\'\n        return finish_supsign_preparation(source, output, rows=rows, report=report,\n            excluded=excluded, duplicates_removed=duplicates_removed, settings=settings, progress=progress)\n\n\ndef finish_supsign_preparation(source, output, *, rows, report, excluded,\n                               duplicates_removed, settings, progress=None):\n    """Finish an audited/grouped snapshot; verify copied image hashes before publishing."""\n    from copy import deepcopy\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output.exists(): raise FileExistsError(output)\n    rows = deepcopy(rows)  # Keep failed-frame state available if finalization needs retry.\n    classes, seed = settings[\'classes\'], settings[\'seed\']\n    counts, missing = choose_splits(rows,classes,seed)\n    with DatasetReader(source) as reader:\n        if reader.warnings: raise ValueError(\'Source archive warnings: \'+str(reader.warnings[:3]))\n        output.parent.mkdir(parents=True,exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix=\'.supsign-\',dir=output.parent) as temporary:\n            stage = Path(temporary)/\'dataset\';stage.mkdir()\n            writing = progress(rows, desc=\'Write prepared images\', unit=\'image\') if progress else rows\n            for row in writing:\n                suffix = PurePosixPath(row[\'source_path\']).suffix.lower()\n                row[\'image\'] = f"images/{row[\'split\']}/{row[\'sha256\']}{suffix}"\n                row[\'label\'] = f"labels/{row[\'split\']}/{row[\'sha256\']}.txt"\n                image, label = stage/row[\'image\'], stage/row[\'label\']\n                image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n                with reader.open(row[\'source_path\']) as stream, image.open(\'wb\') as target:shutil.copyfileobj(stream,target)\n                if digest_file(image) != row[\'sha256\']:raise ValueError(\'Source image changed during preparation\')\n                label.write_text(\'\\n\'.join(row.pop(\'lines\'))+\'\\n\',encoding=\'utf-8\')\n                row[\'label_sha256\'] = digest_file(label);row.pop(\'visual_hash\')\n            manifest = dict(kind=\'supsign_car35_experiment\',recipe=settings,names=list(classes),\n                source_zip_sha256=digest_file(source),source_summary=report[\'summary\'],\n                images=rows,excluded=excluded,issues=report[\'issues\'],duplicates_removed=duplicates_removed,\n                class_counts={s:dict(counts[s]) for s in SPLITS},missing_classes=missing,\n                split_provenance=report.get(\'_split_provenance\', \'development_heuristic\'),\n                independent_field_test=False,\n                limitations=\'Exact-byte duplicates removed; dHash and filename blocks are heuristics. \'\n                            \'An independent car/windshield field test is still required. \'\n                            \'Source-negative/other-class images are negatives only for the selected taxonomy.\')\n            write_json(stage/\'manifest.json\',manifest)\n            config=dict(path=\'.\',train=\'images/train\',val=\'images/val\',test=\'images/test\',names=list(classes))\n            (stage/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n            stage.rename(output)\n    return manifest\n\n\ndef recover_failed_preparation(traceback, *, progress=None):\n    """Resume the split failure in the current Python session, without re-auditing.\n\n    The traceback must retain prepare_supsign\'s fully audited and grouped rows.\n    Never accepts an arbitrary saved file as proof that the source was audited.\n    """\n    frame = None\n    while traceback is not None:\n        candidate = traceback.tb_frame\n        if (candidate.f_code.co_name == \'prepare_supsign\' and\n            Path(candidate.f_code.co_filename).name == \'supsign_training.py\'):\n            state = candidate.f_locals\n            if (state.get(\'rows\') and all(\'group\' in row and \'lines\' in row for row in state[\'rows\'])\n                and all(key in state for key in (\'source\',\'output\',\'report\',\'excluded\',\'duplicates_removed\'))):\n                frame = dict(state)\n        traceback = traceback.tb_next\n    if frame is None:\n        raise ValueError(\'No completed preparation snapshot in traceback; keep the original failed runtime \'\n                         \'or rerun preparation with the updated notebook\')\n    def inventory(reader):\n        return {name: (member.CRC, member.file_size) for name, (_, member, _) in reader.entries.items()\n                if member is not None}\n    original = frame.get(\'reader\')\n    if original is None: raise ValueError(\'Missing source inventory in preparation traceback\')\n    with DatasetReader(frame[\'source\']) as current:\n        if inventory(original) != inventory(current):\n            raise ValueError(\'Source ZIP changed since the failed preparation; rerun the audit\')\n    report = dict(frame[\'report\'])\n    report[\'_split_provenance\'] = \'provided_capture_groups\' if frame.get(\'groups\') is not None else \'development_heuristic\'\n    settings = recipe(frame[\'classes\'], frame[\'seed\'], frame[\'block_size\'],\n                      frame[\'near_threshold\'], frame.get(\'groups\'))\n    return finish_supsign_preparation(frame[\'source\'], frame[\'output\'], rows=frame[\'rows\'],\n        report=report, excluded=frame[\'excluded\'], duplicates_removed=frame[\'duplicates_removed\'],\n        settings=settings, progress=progress)\n\n\ndef verify_dataset(root):\n    root = Path(root).resolve();manifest = read_json(root/\'manifest.json\')\n    config = yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config != dict(path=\'.\',train=\'images/train\',val=\'images/val\',test=\'images/test\',names=manifest[\'names\']):\n        raise ValueError(\'Dataset YAML changed\')\n    hashes,groups,expected,labels = set(),{},set(),set()\n    for row in manifest[\'images\']:\n        image,label = root/row[\'image\'],root/row[\'label\'];split=row[\'split\']\n        if split not in SPLITS or image.parent != root/\'images\'/split or label != root/\'labels\'/split/(image.stem+\'.txt\'):\n            raise ValueError(\'Unsafe or misplaced manifest path\')\n        if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=split):\n            raise ValueError(\'Duplicate image or cross-split capture group\')\n        if digest_file(image)!=row[\'sha256\'] or digest_file(label)!=row[\'label_sha256\']:raise ValueError(\'Dataset image/label changed\')\n        hashes.add(row[\'sha256\']);groups[row[\'group\']]=split;expected.add(image);labels.add(label)\n    if expected!=set((root/\'images\').rglob(\'*.*\')) or labels!=set((root/\'labels\').rglob(\'*.txt\')):\n        raise ValueError(\'Dataset inventory changed\')\n    if {r[\'split\'] for r in manifest[\'images\']}!=set(SPLITS):raise ValueError(\'Empty dataset split\')\n    return dict(images=len(expected),names=manifest[\'names\'],manifest_sha256=digest_file(root/\'manifest.json\'),\n                independent_field_test=manifest[\'independent_field_test\'],split_provenance=manifest[\'split_provenance\'])\n\n\ndef summarize_prepared_dataset(root, *, imgsz=640):\n    """EDA of the exact prepared training set, without changing images/labels/splits."""\n    root = Path(root).resolve()\n    if type(imgsz) is not int or imgsz < 32:\n        raise ValueError(\'Invalid EDA input image size\')\n    manifest = read_json(root / \'manifest.json\')\n    names = manifest[\'names\']\n    images, boxes = [], []\n    counts = {split: Counter() for split in SPLITS}\n    negatives = 0\n    for row in manifest[\'images\']:\n        # Preparation/cache verification already checked hashes and decoded images.\n        with Image.open(root / row[\'image\']) as image:\n            width, height = image.size\n        entries = [line.split() for line in (root / row[\'label\']).read_text().splitlines() if line.strip()]\n        if not entries:\n            negatives += 1\n        images.append(dict(image=row[\'image\'], split=row[\'split\'], group=row[\'group\'],\n                           width=width, height=height, boxes=len(entries)))\n        for entry in entries:\n            cls = int(entry[0]); _, _, w, h = map(float, entry[1:])\n            counts[row[\'split\']][cls] += 1\n            scale = imgsz / max(width, height)\n            boxes.append(dict(image=row[\'image\'], split=row[\'split\'], class_id=cls, name=names[cls],\n                              width_px=w*width, height_px=h*height, area_fraction=w*h,\n                              aspect_ratio=w*width/(h*height),\n                              short_side_at_input=min(w*width,h*height)*scale))\n    catalogue = [dict(class_id=i, name=name, **{s: counts[s][i] for s in SPLITS})\n                 for i, name in enumerate(names)]\n    exclusions = Counter(row[\'reason\'] for row in manifest[\'excluded\'])\n    summary = dict(images=len(images), boxes=len(boxes), negative_images=negatives,\n                   groups=len({r[\'group\'] for r in images}), input_size=imgsz,\n                   boxes_under_16px=sum(b[\'short_side_at_input\'] < 16 for b in boxes),\n                   duplicates_removed=manifest[\'duplicates_removed\'], excluded_images=len(manifest[\'excluded\']),\n                   exclusion_reasons=dict(exclusions), missing_classes=manifest[\'missing_classes\'],\n                   split_provenance=manifest[\'split_provenance\'], independent_field_test=False,\n                   source_summary=manifest[\'source_summary\'], limitations=manifest[\'limitations\'])\n    return dict(summary=summary, class_counts=catalogue, images=images, boxes=boxes)\n\n\ndef pack_dataset(root,target):\n    root,target=Path(root).resolve(),Path(target).resolve()\n    if target.is_relative_to(root):raise ValueError(\'Dataset cache must be outside the dataset\')\n    if target.exists() or target.with_suffix(target.suffix+\'.sha256\').exists():raise FileExistsError(target)\n    verify_dataset(root);target.parent.mkdir(parents=True,exist_ok=True)\n    temporary=target.with_suffix(target.suffix+\'.part\')\n    if temporary.exists():raise FileExistsError(temporary)\n    try:\n        with zipfile.ZipFile(temporary,\'x\',compression=zipfile.ZIP_STORED) as archive:\n            for path in sorted(root.rglob(\'*\')):\n                if path.is_file():archive.write(path,path.relative_to(root).as_posix())\n        temporary.rename(target)\n        target.with_suffix(target.suffix+\'.sha256\').write_text(digest_file(target),encoding=\'utf-8\')\n    finally:\n        if temporary.exists():temporary.unlink()\n    return target\n\n\ndef restore_dataset(source,output,*,expected_recipe=None):\n    source,output=Path(source),Path(output).resolve()\n    if output.exists():raise FileExistsError(output)\n    if digest_file(source)!=source.with_suffix(source.suffix+\'.sha256\').read_text().strip():\n        raise ValueError(\'Dataset cache checksum mismatch\')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(source) as archive, tempfile.TemporaryDirectory(prefix=\'.restore-\',dir=output.parent) as temporary:\n        stage=Path(temporary)/\'dataset\';stage.mkdir();seen=set()\n        for member in archive.infolist():\n            relative=PurePosixPath(member.filename)\n            if (relative.is_absolute() or \'..\' in relative.parts or \'\\\\\' in member.filename\n                    or \':\' in member.filename or member.filename in seen\n                    or (member.external_attr>>16)&0o170000==0o120000):raise ValueError(\'Unsafe cache ZIP path\')\n            seen.add(member.filename)\n            if member.is_dir():continue\n            path=stage/relative;path.parent.mkdir(parents=True,exist_ok=True)\n            with archive.open(member) as stream,path.open(\'wb\') as target:shutil.copyfileobj(stream,target)\n        manifest=read_json(stage/\'manifest.json\')\n        if expected_recipe is not None and manifest[\'recipe\']!=expected_recipe:raise ValueError(\'Cache recipe changed; choose a new cache version\')\n        verify_dataset(stage);stage.rename(output)\n    return manifest\n\n\ndef train_experiment(root,run_dir,*,epochs=80,imgsz=640,batch=8,device=\'0\',resume=False,yolo_factory=None):\n    root,run_dir=Path(root).resolve(),Path(run_dir).resolve()\n    if run_dir.is_relative_to(root):raise ValueError(\'Training run must be outside dataset\')\n    if epochs<1 or imgsz<32 or batch<1:raise ValueError(\'Invalid training settings\')\n    dataset=verify_dataset(root)\n    request_path=run_dir/\'training_request.json\'\n    settings=dict(model=\'yolo11s.pt\',epochs=epochs,imgsz=imgsz,batch=batch)\n    if resume:\n        if not (run_dir/\'weights/last.pt\').is_file() or not request_path.is_file():raise ValueError(\'Resume requires saved last.pt and training_request.json\')\n        previous=read_json(request_path)\n        if previous[\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Resume dataset changed\')\n        if previous[\'settings\']!=settings:raise ValueError(\'Resume settings changed; use the original settings or a new run\')\n        model=run_dir/\'weights/last.pt\'\n    else:\n        if run_dir.exists():raise FileExistsError(\'Run exists; set RESUME=True or choose another RUN_NAME\')\n        model=\'yolo11s.pt\'\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(model))\n    if not resume:\n        run_dir.mkdir(parents=True)\n        write_json(request_path,dict(**dataset,settings=settings,test_evaluated=False))\n    # Dataset archive remains portable; runtime absolute path lives beside the run.\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'));config[\'path\']=str(root)\n    runtime_data=run_dir/\'data_runtime.yaml\';runtime_data.write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n    options=dict(data=str(runtime_data),epochs=epochs,imgsz=imgsz,batch=batch,device=device,\n                 project=str(run_dir.parent),name=run_dir.name,exist_ok=True,workers=2,\n                 seed=42,deterministic=True,cache=False,amp=True,patience=20,save=True,save_period=5,\n                 fliplr=0.,flipud=0.,degrees=0.,shear=0.,perspective=0.,mixup=0.,\n                 mosaic=.5,close_mosaic=min(10,epochs),plots=True)\n    if resume:options[\'resume\']=True\n    detector.train(**options)\n    if not (run_dir/\'weights/best.pt\').is_file():raise ValueError(\'Training did not produce best.pt\')\n    return run_dir/\'weights/best.pt\'\n\n\ndef evaluate_checkpoint(checkpoint,root,output,*,split=\'val\',allow_test=False,imgsz=640,batch=8,yolo_factory=None):\n    checkpoint,root,output=Path(checkpoint),Path(root).resolve(),Path(output).resolve()\n    if split not in (\'val\',\'test\') or split==\'test\' and not allow_test:raise ValueError(\'Held-out test must be explicitly enabled after choosing settings\')\n    if output.exists():raise FileExistsError(output)\n    if output.is_relative_to(root):raise ValueError(\'Evaluation must be outside dataset\')\n    dataset=verify_dataset(root)\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(checkpoint))\n    names=detector.names\n    names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Checkpoint taxonomy differs from dataset\')\n    output.mkdir(parents=True)\n    config=yaml.safe_load((root/\'data.yaml\').read_text());config[\'path\']=str(root)\n    runtime_data=output/\'data_runtime.yaml\';runtime_data.write_text(yaml.safe_dump(config),encoding=\'utf-8\')\n    result=detector.val(data=str(runtime_data),split=split,imgsz=imgsz,batch=batch,\n                        project=str(output.parent),name=output.name,exist_ok=True,plots=True)\n    report=dict(**dataset,split=split,checkpoint_sha256=digest_file(checkpoint),\n                metrics={k:float(v) if math.isfinite(float(v)) else None for k,v in result.results_dict.items()})\n    box=getattr(result,\'box\',None)\n    if box is not None:\n        report[\'per_class\']=[dict(class_id=int(i),name=names[int(i)],precision=float(box.p[j]),\n                                 recall=float(box.r[j]),map50=float(box.ap50[j]),map50_95=float(box.ap[j]))\n                             for j,i in enumerate(box.ap_class_index)]\n        report[\'classes_without_gt\']=[n for i,n in enumerate(names) if i not in set(box.ap_class_index)]\n    write_json(output/\'evaluation_report.json\',report)\n    return report\n'}
CODE_ROOT = WORK / 'pipeline'
for relative, source in TRAINING_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
(CODE_ROOT / 'scripts/__init__.py').touch()
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from scripts.supsign_training import (CAR35, recipe, prepare_supsign, verify_dataset,
    digest_file, pack_dataset, restore_dataset, train_experiment, evaluate_checkpoint,
    summarize_prepared_dataset)
GROUPS = None
if GROUPS_CSV is not None:
    with Path(GROUPS_CSV).open(newline='', encoding='utf-8-sig') as stream:
        group_rows = list(csv.DictReader(stream))
    assert all(row.get('path') and row.get('group') for row in group_rows), 'CSV cần path,group.'
    GROUPS = {row['path']: row['group'] for row in group_rows}
    assert len(GROUPS) == len(group_rows), 'CSV có đường dẫn trùng.'
RECIPE = recipe(CAR35, SEED, FRAME_BLOCK_SIZE, NEAR_HASH_DISTANCE, GROUPS)
print('Taxonomy:', len(CAR35), 'lớp. Cache sẽ kiểm tra cùng recipe/code và checksum.')

Taxonomy: 35 lớp. Cache sẽ kiểm tra cùng recipe/code và checksum.


## 2. Chuẩn bị dataset hoặc khôi phục cache

Lần đầu đọc/hash toàn bộ ZIP có thể mất thời gian. Cần đủ ổ local cho ZIP và dataset đã lọc,
và đủ Drive để giữ cache + các run. Không giải nén ZIP nguồn tùy ý; chỉ ghi ảnh/nhãn đã kiểm tra.
Sau khi có cache trên Drive, runtime mới không cần copy hoặc audit lại ZIP thô.
Cache có checksum; thay code/classes/grouping thì phải dùng một DATASET_VERSION mới.
V3 giữ đủ lớp trong train nếu có thể tạo ba split không rỗng mà không tách nhóm.
Lớp chỉ có trong một nhóm được giữ ở train; val/test có thể thiếu lớp đó và được ghi rõ.

Nhãn lỗi bị cách ly cả ảnh, không lặng lẽ sửa annotation nguồn. Ảnh trùng byte nhưng nhãn khác
cũng bị cách ly để xem lại. Ảnh chỉ chứa lớp ngoài scope được giữ làm nền đối với 35 lớp mục tiêu.

In [4]:
from tqdm.auto import tqdm
if DATASET_ROOT.exists():
    info = verify_dataset(DATASET_ROOT)
    manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text())
    assert manifest['recipe'] == RECIPE, 'Đổi DATASET_VERSION: dataset local dùng recipe khác.'
    print('Dùng dataset local đã kiểm tra:', info['images'], 'ảnh')
elif CACHE_ZIP.is_file():
    manifest = restore_dataset(CACHE_ZIP, DATASET_ROOT, expected_recipe=RECIPE)
    print('Đã khôi phục cache Drive, không đọc lại ZIP thô.')
else:
    assert SOURCE_ZIP.is_file(), f'Không thấy ZIP: {SOURCE_ZIP}. Sửa đường dẫn ở cell Cấu hình.'
    required = SOURCE_ZIP.stat().st_size * 2.3 + 2*1024**3
    free = shutil.disk_usage(WORK).free
    assert free >= required, f'Ổ local cần khoảng {required/1024**3:.1f} GiB trống; còn {free/1024**3:.1f}.'
    local_zip = WORK / 'source_supsign.zip'
    # Copy thẳng từ Drive, không dùng upload lại qua trình duyệt.
    shutil.copy2(SOURCE_ZIP, local_zip)
    manifest = prepare_supsign(local_zip, DATASET_ROOT, classes=CAR35, groups=GROUPS,
        seed=SEED, block_size=FRAME_BLOCK_SIZE, near_threshold=NEAR_HASH_DISTANCE,
        progress=lambda items, desc, unit: tqdm(items, desc=desc, unit=unit))
    print('Đang kiểm tra và lưu cache dataset lên Drive...', flush=True)
    pack_dataset(DATASET_ROOT, CACHE_ZIP)
    local_zip.unlink()  # chỉ xóa bản sao local do cell này tạo, không xóa ZIP trên Drive
    print('Đã lưu cache Drive:', CACHE_ZIP)
from collections import Counter
split_counts = Counter(row['split'] for row in manifest['images'])
print(' | '.join(f'{s}: {split_counts[s]:,} ảnh' for s in ('train', 'val', 'test')))
print('Loại bản sao:', manifest['duplicates_removed'], '| Cách ly:', len(manifest['excluded']))
print('Split:', manifest['split_provenance'], '| Test thực địa độc lập:', manifest['independent_field_test'])
for split, missing in manifest['missing_classes'].items():
    if missing: print('Lớp thiếu ở', split, ':', ', '.join(missing))

AssertionError: Không thấy ZIP: /content/drive/MyDrive/SignSight/datasets/archive (1).zip. Sửa đường dẫn ở cell Cấu hình.

## EDA trước khi train — lưu báo cáo trên Drive

Thống kê **đúng dataset 35 lớp sau khi lọc**, kèm audit nguồn trong `source_summary`/manifest.
Xuất JSON + CSV (ảnh, bbox, số mẫu từng lớp), biểu đồ phân bố lớp/split, kích thước ảnh,
diện tích và cạnh ngắn bbox ở kích thước train. `boxes_under_16px` giúp nhận ra biển nhỏ khó học.
Không audit lại ZIP thô nếu đã có cache. Chỉ in tóm tắt; chi tiết lỗi/cách ly nằm trong manifest.
Nhóm filename/dHash vẫn là heuristic; EDA không thay tập kiểm tra qua kính ô tô độc lập.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
eda = summarize_prepared_dataset(DATASET_ROOT, imgsz=IMAGE_SIZE)
EDA_ROOT = PERSIST / 'eda' / DATASET_VERSION
EDA_ROOT.mkdir(parents=True, exist_ok=True)
(EDA_ROOT / 'dataset_eda.json').write_text(json.dumps(eda['summary'], ensure_ascii=False, indent=2), encoding='utf-8')
for key in ('class_counts', 'images', 'boxes'):
    rows = eda[key]
    with (EDA_ROOT / (key + '.csv')).open('w', newline='', encoding='utf-8-sig') as stream:
        if rows:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
y = list(range(len(eda['class_counts']))); starts = [0] * len(y)
for split in ('train', 'val', 'test'):
    values = [r[split] for r in eda['class_counts']]
    axes[0,0].barh(y, values, left=starts, label=split)
    starts = [a+b for a,b in zip(starts, values)]
axes[0,0].set_yticks(y, [r['name'] for r in eda['class_counts']], fontsize=8)
axes[0,0].set_title('Boxes per class / split'); axes[0,0].legend()
axes[0,1].hist([r['short_side_at_input'] for r in eda['boxes']], bins=40)
axes[0,1].axvline(16, color='red', linestyle='--'); axes[0,1].set_title(f'BBox short side at input {IMAGE_SIZE} (px)')
axes[1,0].scatter([r['width'] for r in eda['images']], [r['height'] for r in eda['images']], s=3, alpha=.15)
axes[1,0].set(xlabel='Image width', ylabel='Image height', title='Image dimensions')
axes[1,1].hist([r['area_fraction'] for r in eda['boxes']], bins=40)
axes[1,1].set_title('BBox area / image area')
fig.tight_layout(); fig.savefig(EDA_ROOT / 'eda_overview.png', dpi=150); plt.show(); plt.close(fig)
summary = eda['summary']
print(f"EDA: {summary['images']:,} images | {summary['boxes']:,} boxes | {summary['negative_images']:,} negatives")
print(f"Small boxes (<16 px at {IMAGE_SIZE}): {summary['boxes_under_16px']:,}")
print('Duplicate copies removed:', summary['duplicates_removed'], '| Quarantined:', summary['excluded_images'])
print('Saved:', EDA_ROOT)

## 3. Kiểm tra nhãn và phân bố lớp

Xem ảnh đầy đủ, bbox và tên lớp trước khi train. Phân biệt cấm ô tô/cấm chung, cấm đỗ/cấm dừng,
biển hướng trái/phải. Class ID mới là 0–34 theo CAR35; **không dùng ID này thay trực tiếp vào database 10 lớp**.
Các lớp thiếu val/test phải được ghi nhận khi đọc metric; chưa có nhãn không có nghĩa model đạt chất lượng.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random
sample_rows = manifest['images'].copy()
random.Random(SEED).shuffle(sample_rows)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis in axes.flat: axis.axis('off')
for axis, row in zip(axes.flat, sample_rows[:6]):
    with Image.open(DATASET_ROOT / row['image']) as source: image = source.convert('RGB')
    draw = ImageDraw.Draw(image)
    for line in (DATASET_ROOT / row['label']).read_text().splitlines():
        if not line.strip(): continue
        cls, x, y, w, h = map(float, line.split())
        box = ((x-w/2)*image.width, (y-h/2)*image.height, (x+w/2)*image.width, (y+h/2)*image.height)
        draw.rectangle(box, outline='yellow', width=2)
        draw.text((max(0, box[0]), max(0, box[1]-12)), CAR35[int(cls)], fill='yellow')
    axis.imshow(image); axis.set_title(row['split'])
plt.tight_layout(); plt.show()
# Gọn 35 dòng để thấy lớp ít mẫu; không in hàng chục nghìn đường dẫn.
print(f"{'Class':35s} {'train':>7s} {'val':>7s} {'test':>7s}")
for i, name in enumerate(CAR35):
    values = [manifest['class_counts'][s].get(str(i), manifest['class_counts'][s].get(i, 0)) for s in ('train', 'val', 'test')]
    print(f'{name:35s} {values[0]:7d} {values[1]:7d} {values[2]:7d}')

## 4. Train YOLO11s

Tải pretrained **yolo11s.pt**, không tiếp tục model GTSRB cũ. Tắt lật ngang/dọc và xoay ảnh để giữ nghĩa biển.
Checkpoint, results.csv, cấu hình và plots được ghi vào RUN_DIR trên Drive, còn ảnh train ở local Colab.
Best được chọn theo validation. Cell này **không đánh giá test**.

Nếu runtime ngắt giữa chừng: mở lại notebook, giữ RUN_NAME/EPOCHS/IMAGE_SIZE/BATCH,
đổi RESUME=True rồi chạy lại các cell. Checkpoint last.pt giữ trạng thái optimizer để resume;
run đã hoàn tất có thể không còn resume được. Run mới phải có tên mới.

In [ ]:
assert LABEL_REVIEWED, 'Xem EDA và ảnh bbox, rồi đổi LABEL_REVIEWED=True ở cell Cấu hình.'
BEST = train_experiment(DATASET_ROOT, RUN_DIR,
    epochs=1 if SMOKE else EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH, device='0', resume=RESUME)
print('Best checkpoint:', BEST)

## 5. Validation và test có chủ đích

Chọn cấu hình bằng validation; so sánh precision/recall/mAP từng lớp, không chỉ mAP tổng.
Giữ RUN_HELDOUT_TEST=False trong lúc chọn model. Khi bật True, kết quả test vẫn ghi rõ đây là split
của dataset, chưa phải một tập thu thập thực địa độc lập. Không dùng test để điều chỉnh confidence.
Chọn EVAL_NAME mới nếu muốn lưu một lượt đánh giá khác.

In [ ]:
EVAL_NAME = 'evaluation_v1'
validation = evaluate_checkpoint(BEST, DATASET_ROOT, RUN_DIR / EVAL_NAME / 'val',
                                 imgsz=IMAGE_SIZE, batch=BATCH)
print('Validation:', json.dumps(validation['metrics'], indent=2))
if RUN_HELDOUT_TEST:
    test_report = evaluate_checkpoint(BEST, DATASET_ROOT, RUN_DIR / EVAL_NAME / 'test',
        split='test', allow_test=True, imgsz=IMAGE_SIZE, batch=BATCH)
    print('Dataset test:', json.dumps(test_report['metrics'], indent=2))
else:
    print('Chưa chạy test. Chốt cấu hình bằng val trước khi bật RUN_HELDOUT_TEST.')
from IPython.display import display
for name in ('results.png', EVAL_NAME + '/val/confusion_matrix.png'):
    plot = RUN_DIR / name
    if plot.is_file(): display(Image.open(plot))

## 6. Thử ảnh đường phố riêng (tùy chọn)

Upload vài ảnh qua kính xe / ảnh không có biển / biển xa để xem bbox. Đây là kiểm tra trực quan,
không thay phép đánh giá với ground truth. Confidence xem ảnh không thay metric mAP của validation.

In [ ]:
TRY_OWN_PHOTOS = False
if TRY_OWN_PHOTOS:
    from google.colab import files
    from ultralytics import YOLO
    uploaded = files.upload()
    photo_root = WORK / 'own_photos'; photo_root.mkdir(exist_ok=True)
    paths = []
    for filename, content in uploaded.items():
        target = photo_root / Path(filename).name
        target.write_bytes(content); paths.append(str(target))
    for result in YOLO(str(BEST)).predict(paths, conf=0.25, imgsz=IMAGE_SIZE, save=False):
        display(Image.fromarray(result.plot()[..., ::-1]))

## 7. Tải ZIP kết quả

ZIP gồm best.pt, last.pt, cấu hình train, metric, plots và manifest dataset; **không kèm toàn bộ ảnh dataset**.
Đã có bản trên Drive nếu trình duyệt tải chậm. Dùng files.download để tránh link localhost không tải được.

Model 35 lớp là artifact thử nghiệm. Backend SignSight hiện nhận taxonomy 10 lớp: cần nâng mapping,
validation và kiểm thử tương thích dữ liệu cũ trước khi thay checkpoint phục vụ app.

In [ ]:
from datetime import datetime, timezone
from google.colab import files
export_name = RUN_NAME + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
export_root = WORK / 'exports' / export_name
export_root.mkdir(parents=True, exist_ok=False)
shutil.copytree(RUN_DIR, export_root / 'run')
if 'EDA_ROOT' in globals() and EDA_ROOT.is_dir(): shutil.copytree(EDA_ROOT, export_root / 'eda')
metadata = export_root / 'dataset'; metadata.mkdir()
for name in ('data.yaml', 'manifest.json'): shutil.copy2(DATASET_ROOT / name, metadata / name)
(metadata / 'classes.json').write_text(json.dumps(CAR35, indent=2), encoding='utf-8')
archive = Path(shutil.make_archive(str(export_root), 'zip', export_root))
exports_drive = PERSIST / 'exports'; exports_drive.mkdir(exist_ok=True)
drive_export = exports_drive / archive.name
shutil.copy2(archive, drive_export)
print('Drive:', drive_export)
files.download(str(archive))